In [15]:
import pennylane as qml
from pennylane import numpy as np

In [16]:
molecule = ['H','H']
coordinates = np.array([0.0, 0.0, -0.6614, 0.0, 0.0, 0.6614])

In [17]:
hamil,qubits = qml.qchem.molecular_hamiltonian(molecule, coordinates)
electrons = 2

In [18]:
hartree_fock_state = qml.qchem.hf_state(electrons,qubits)
print(f'hartree fock state is {hartree_fock_state}')

hartree fock state is [1 1 0 0]


In [19]:
single,double = qml.qchem.excitations(electrons,qubits)
print(f'single excitations are {single}')
print(f'double excitation are {double}')

single excitations are [[0, 2], [1, 3]]
double excitation are [[0, 1, 2, 3]]


In [26]:
device = qml.device('default.qubit',wires=4)
def quantum_circuit(theta):
    qml.BasisState(hartree_fock_state,wires = range(4))
    for i,s_wires in enumerate(single):
        qml.SingleExcitation(theta[i],wires=s_wires)
    for i,d_wires in enumerate(double):
        qml.DoubleExcitation(theta[len(single)+i],wires=d_wires)
    return qml.expval(hamil)
circuit = qml.QNode(quantum_circuit,device)

In [33]:
theta=np.array([0.0,0.0,0.0],requires_grad = True)
optimiser = qml.GradientDescentOptimizer(stepsize=0.4)
iterations = 40
for i in range(iterations):
    theta,energy = optimiser.step_and_cost(circuit,theta)
    if i%10==0:
        print(f'theta is {theta} and previous energy is {energy}')
print()
print(f'final theta is {theta}')
print(f'ground state energy is {circuit(theta)}')

theta is [0.         0.         0.07160013] and previous energy is -1.117348921135931
theta is [0.         0.         0.20768527] and previous energy is -1.1361849754890532
theta is [0.         0.         0.20970262] and previous energy is -1.1361891616071993
theta is [0.         0.         0.20973244] and previous energy is -1.1361891625216802

final theta is [0.         0.         0.20973288]
ground state energy is -1.1361891625218803
